# **Clean and Prep ADA Race Results v2**

Cleans and standardizes 5K@ADA race results by removing duplicates and translating gender values into English. Participant names are used briefly for deduplication and then discarded to maintain privacy.

In [53]:
# Import packages

# For data manipulation
import pandas as pd

# for displaying and modifying the working directory
import os as os

In [54]:
# Define custom NA values, excluding 'NA' to make sure that Country Code NA (Namibia) is not interpreted as missing data
custom_na_values = ['N/A', 'NaN', 'null', '']

In [55]:
#Load specific columns, ignore the default NA values and use the custom ones
df0 = pd.read_csv("ADA Race Results.csv", usecols=['Bib', 'Country', 'Time', 'Age Group', 'Gender'], keep_default_na=False, na_values=custom_na_values)

# Display the first 5 rows of the dataframe
df0.head()

,Bib,Gender,Country,Time,Age Group
0,416,M,BD,00:14:45,M2024
1,3066,F,PH,00:14:45,F4549
2,6329,M,KE,00:14:45,M2529
3,7531,M,IN,00:14:45,M2024
4,7556,M,BD,00:14:45,M019


In [56]:
# Display basic information about the data 
df0.info()

<class 'pandas.DataFrame'>
RangeIndex: 4052 entries, 0 to 4051
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Bib        4052 non-null   int64
 1   Gender     4038 non-null   str  
 2   Country    4052 non-null   str  
 3   Time       4052 non-null   str  
 4   Age Group  4032 non-null   str  
dtypes: int64(1), str(4)
memory usage: 158.4 KB


In [57]:
# Count number of duplicate rows based on Bib
duplicate_count = df0.duplicated(subset='Bib').sum()
print(f"Number of duplicate rows based on Bib: {duplicate_count}")

Number of duplicate rows based on Bib: 0


In [58]:
# Delete duplicate based on Bib
df0 = df0.drop_duplicates(subset='Bib')

In [59]:
# Verify that duplicates have been removed
duplicate_count_after = df0.duplicated(subset='Bib').sum()
print(f"Number of duplicate rows based on Bib after removal: {duplicate_count_after}")

Number of duplicate rows based on Bib after removal: 0


In [60]:
# Display the number of missing values in each column
df0.isnull().sum()

Bib           0
Gender       14
Country       0
Time          0
Age Group    20
dtype: int64

In [61]:
# Fill missing values in the 'Gender' column and standardize gender labels
df0['Gender'] = df0['Gender'].fillna('Unknown').replace({
    'F': 'Female',
    'M': 'Male',
    'X': 'Unknown'
})

In [62]:
# Add a column for the year
df0['Year'] = 2026

In [63]:
# Add a column for the race name
df0['Race'] = 'ADA'

In [64]:
# Rename Country to Country Code
df0 = df0.rename(columns={'Country': 'Country Code'})

In [65]:
# Change the column order
df0 = df0[['Race', 'Year', 'Country Code', 'Age Group', 'Time', 'Gender', 'Bib']]

In [66]:
# Replace null values in the 'Age Group' column with 'Unknown'
df0.fillna({'Age Group': 'Unknown'}, inplace=True)

,Race,Year,Country Code,Age Group,Time,Gender,Bib
0,ADA,2026,BD,M2024,00:14:45,Male,416
1,ADA,2026,PH,F4549,00:14:45,Female,3066
2,ADA,2026,KE,M2529,00:14:45,Male,6329
3,ADA,2026,IN,M2024,00:14:45,Male,7531
4,ADA,2026,BD,M019,00:14:45,Male,7556
...,...,...,...,...,...,...,...
4047,ADA,2026,KE,F3539,02:00:00,Female,13738
4048,ADA,2026,ZM,F4549,02:00:00,Female,15619
4049,ADA,2026,ID,F4044,02:00:00,Female,2506
4050,ADA,2026,ZA,M3034,02:00:00,Male,7228


In [67]:
# Standardize age groups
df0['Age Group'] = df0['Age Group'].replace(
    r'^[MF](\d{1,2})(\d{2})$',
    r'\1-\2',
    regex=True
)

In [68]:
# Display count of duplicate rows
df0.duplicated().sum()

np.int64(0)

In [69]:
# Delete duplicate rows
df0 = df0.drop_duplicates()

In [70]:
# Display basic information about the data
df0.info()

<class 'pandas.DataFrame'>
RangeIndex: 4052 entries, 0 to 4051
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Race          4052 non-null   str  
 1   Year          4052 non-null   int64
 2   Country Code  4052 non-null   str  
 3   Age Group     4052 non-null   str  
 4   Time          4052 non-null   str  
 5   Gender        4052 non-null   str  
 6   Bib           4052 non-null   int64
dtypes: int64(2), str(5)
memory usage: 221.7 KB


In [71]:
# Drop the Name column
df0 = df0.drop(columns=['Bib'])

In [72]:
# If Age Group or Gender is 'Unknown', drop the row
df0 = df0[(df0['Age Group'] != 'Unknown') & (df0['Gender'] != 'Unknown')]

In [73]:
df0 = df0.sort_values(by=["Race", "Year", "Time"])

In [74]:
# Display the number of missing values in each column
df0.isnull().sum()

Race            0
Year            0
Country Code    0
Age Group       0
Time            0
Gender          0
dtype: int64

In [75]:
# Reset index
df0 = df0.reset_index(drop=True)

In [76]:
# Verify the change
df0.head()

,Race,Year,Country Code,Age Group,Time,Gender
0,ADA,2026,BD,20-24,00:14:45,Male
1,ADA,2026,PH,45-49,00:14:45,Female
2,ADA,2026,KE,25-29,00:14:45,Male
3,ADA,2026,IN,20-24,00:14:45,Male
4,ADA,2026,BD,0-19,00:14:45,Male


In [77]:
# Display basic information about the data 
df0.info()

<class 'pandas.DataFrame'>
RangeIndex: 4032 entries, 0 to 4031
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Race          4032 non-null   str  
 1   Year          4032 non-null   int64
 2   Country Code  4032 non-null   str  
 3   Age Group     4032 non-null   str  
 4   Time          4032 non-null   str  
 5   Gender        4032 non-null   str  
dtypes: int64(1), str(5)
memory usage: 189.1 KB


In [78]:
# Save the cleaned data to a new csv file
df0.to_csv('ADA Race Data for Visualization.csv', index=False)